# TP2 — Nettoyage et tokenisation d'un corpus d'avis
### EMSI — Traitement Automatique du Langage Naturel (TALN)
**Version étudiant**

**Place dans le cours :** Chapitre 2 (2/3) — regex avancées, nettoyage, tokenisation, segmentation (séquence 3, séance 6)
**Bibliothèques :** re, html, emoji, pandas, NLTK, spaCy (fr_core_news_sm)

**Objectifs**
- Repérer et mesurer le bruit d'un corpus réel (HTML, liens, mentions, émojis, allongements, darija), avec ou sans la bibliothèque `emoji`.
- Écrire un nettoyeur par étapes avec les regex avancées (groupes, références arrière, assertions) et justifier l'ordre des étapes.
- Comparer plusieurs tokeniseurs (split, regex, NLTK, TweetTokenizer, spaCy) et plusieurs segmenteurs en phrases.
- Assembler une fonction `preprocess()` réutilisable dans les TP suivants (TF-IDF, sentiment…).

## Mode d'emploi
1. **Renseignez votre identification** dans la cellule ci-dessous (nom, prénom, classe, groupe, site EMSI).
2. Exécutez les cellules **dans l'ordre** (`Maj + Entrée`). Les fonctions d'un exercice sont réutilisées dans les suivants.
3. Complétez les cellules marquées `# TODO` / `...  # ← À COMPLÉTER`. Ne modifiez pas les cellules « ✅ Vérification » : elles vous indiquent si votre code est correct.
4. Répondez aux questions **📝 Interprétation** directement dans la cellule de texte prévue (double-clic pour éditer).
5. Enregistrez le notebook sous le nom affiché par la cellule d'identification, puis déposez-le sur la plateforme.

In [2]:
NUM_TP = 2
# 🪪 IDENTIFICATION — À REMPLIR AVANT DE COMMENCER
NOM = "ELHANNAK"        # ex. "EL AMRANI"
PRENOM = "Badr"     # ex. "Salma"
CLASSE = "5IIR"     # ex. "5IIR"
GROUPE = "G2"     # ex. "G3"
SITE = "Maarif"       # un parmi : "Roudani", "Maarif", "Centre", "CFC", "Oranger", "Moulay Youssef"

SITES_EMSI = ["Roudani", "Maarif", "Centre", "CFC", "Oranger", "Moulay Youssef"]
assert NOM and PRENOM and CLASSE and GROUPE, "Complétez NOM, PRENOM, CLASSE et GROUPE."
assert SITE in SITES_EMSI, f"SITE doit être l'un de {SITES_EMSI}"
NOM_FICHIER = f"TP{NUM_TP}_{NOM.upper().replace(' ', '-')}_{PRENOM.capitalize()}_{CLASSE}_{GROUPE}_{SITE.replace(' ', '')}.ipynb"
print(f"Étudiant(e) : {PRENOM} {NOM.upper()} | Classe : {CLASSE} | Groupe : {GROUPE} | EMSI {SITE}")
print("➡️  Nom du fichier à rendre :", NOM_FICHIER)

Étudiant(e) : Badr ELHANNAK | Classe : 5IIR | Groupe : G2 | EMSI Maarif
➡️  Nom du fichier à rendre : TP2_ELHANNAK_Badr_5IIR_G2_Maarif.ipynb


## Installation et préparation de l'environnement
Exécutez la cellule d'installation (une seule fois par machine ; sur Google Colab, à chaque session).

In [3]:
# Installation des bibliothèques (à exécuter une seule fois)
import sys
%pip install -q nltk spacy pandas emoji
!{sys.executable} -m spacy download fr_core_news_sm -q
print("Installation terminée. Si un module reste introuvable : Noyau → Redémarrer, puis réexécuter.")

Note: you may need to restart the kernel to use updated packages.
Installation terminée. Si un module reste introuvable : Noyau → Redémarrer, puis réexécuter.


'c:\Users\Asus\OneDrive' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
# Imports communs, ressources et outil d'auto-vérification
import warnings
warnings.filterwarnings("ignore")
import re, html
import emoji                      # pip install emoji
from collections import Counter
import pandas as pd
pd.set_option("display.max_colwidth", 120)
import nltk
for ressource in ["punkt", "punkt_tab"]:
    nltk.download(ressource, quiet=True)

def verifier(description, test):
    """Affiche ✅ si test() est vrai, ❌ sinon (sans interrompre le notebook)."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {description} — {type(e).__name__} : {e}")
        return
    print(("✅ " if ok else "❌ ") + description)
    return

print("Environnement prêt.")

Environnement prêt.


---
## Exercice 0 ★ — Le corpus fil rouge : repérer le bruit (10 min)

Le corpus du module est un ensemble d'**avis de clients sur des restaurants** au Maroc, collectés sur le Web, sur Twitter/X et par SMS. Ils sont rédigés en français, en darija (souvent en arabizi : *3jbni*, *7ta*) et en arabe. Chaque avis porte une polarité (`pos` / `neg`) qui servira au chapitre 5. Ce corpus est **bruité** : balises HTML, entités (`&amp;`), liens, mentions, hashtags, émojis, lettres répétées…

**Étape 1.** Exécutez la cellule suivante : elle charge le corpus et l'affiche sous forme de tableau.

In [5]:
CORPUS = [
    {"id": 1, "source": "web", "polarite": "pos",
     "texte": "Trooop bon le tajine &amp; le service !!! 😍😍 Je recommande <br>https://avis-resto.ma/r/1542"},
    {"id": 2, "source": "tweet", "polarite": "neg",
     "texte": "@CasaFood livraison en retard de 2h... 😡 #déçu #livraison"},
    {"id": 3, "source": "web", "polarite": "pos",
     "texte": "<p>Accueil <b>impeccable</b>, plats copieux.</p><p>Menu à 85 DH, très correct.</p>"},
    {"id": 4, "source": "sms", "polarite": "pos",
     "texte": "slt, le resto 3jbni bezzaf, l'ambiance mzyana w l'prix m3qol 👍"},
    {"id": 5, "source": "web", "polarite": "neg",
     "texte": "Commande n° 4587 du 03/10/2026 jamais reçue. Contact : service.client@livraison-rapide.ma ... aucune réponse !!!"},
    {"id": 6, "source": "tweet", "polarite": "neg",
     "texte": "Franchement la pizza était froide et sans goût 👎 @PizzaRabat vous pouvez mieux faire #mauvaisservice"},
    {"id": 7, "source": "web", "polarite": "pos",
     "texte": "M. Alami, le gérant, est très aimable. Il nous a offert le thé à la menthe ! Merci."},
    {"id": 8, "source": "sms", "polarite": "neg",
     "texte": "khayb bzf had lmakla, ma3jbnich, 7ta chi haja ma mzyana 😞"},
    {"id": 9, "source": "web", "polarite": "pos",
     "texte": "Rapport qualité/prix : excellent. 2 couscous + 1 jus = 120,50 DH. On reviendra :)"},
    {"id": 10, "source": "tweet", "polarite": "neg",
     "texte": "Attente de 45 min pour une table réservée ?!? Nuuuul. #Casablanca"},
    {"id": 11, "source": "web", "polarite": "pos",
     "texte": "L&#39;équipe est au top, aujourd&#39;hui encore un sans-faute. Les desserts sont faits maison."},
    {"id": 12, "source": "web", "polarite": "neg",
     "texte": "الخدمة بطيئة جدا والأكل بارد. لن أعود مرة أخرى"},
    {"id": 13, "source": "tweet", "polarite": "pos",
     "texte": "Merci @RestoAtlas pour l'anniversaire de ma mère 🎂🎉 Tout était parfait !!! https://t.co/Xy12Ab"},
    {"id": 14, "source": "sms", "polarite": "neg",
     "texte": "Je suis tjrs pas remboursé, ça fait 3 semaines, c'est grave !!"},
    {"id": 15, "source": "web", "polarite": "pos",
     "texte": "Personnel souriant, terrasse agréable. Réservation sur www.terrasse-anfa.ma ou au 06 12 34 56 78."},
    {"id": 16, "source": "tweet", "polarite": "neg",
     "texte": "Prix abusif : 1000 DH pour 4 personnes et des portions miniii 😒 #arnaque"},
    {"id": 17, "source": "web", "polarite": "pos",
     "texte": "Les tajines sont délicieux. Le service est rapide, etc. Bref, une adresse à retenir."},
    {"id": 18, "source": "sms", "polarite": "pos",
     "texte": "wa3er had l'blasa, ghadi nrj3 m3a sshabi inchallah 😍"},
    {"id": 19, "source": "web", "polarite": "neg",
     "texte": "Plats trop salés&nbsp;; le serveur a-t-il seulement goûté ? Bof bof."},
    {"id": 20, "source": "tweet", "polarite": "pos",
     "texte": "Meilleur msemen de la ville !!! 😋👍 @Hanane_K tu dois tester #ftour"},
]

df = pd.DataFrame(CORPUS)
print(len(df), "avis |", df["source"].value_counts().to_dict(), "|", df["polarite"].value_counts().to_dict())
df

20 avis | {'web': 10, 'tweet': 6, 'sms': 4} | {'pos': 11, 'neg': 9}


,id,source,polarite,texte
0,1,web,pos,Trooop bon le tajine &amp; le service !!! 😍😍 Je recommande <br>https://avis-resto.ma/r/1542
1,2,tweet,neg,@CasaFood livraison en retard de 2h... 😡 #déçu #livraison
2,3,web,pos,"<p>Accueil <b>impeccable</b>, plats copieux.</p><p>Menu à 85 DH, très correct.</p>"
3,4,sms,pos,"slt, le resto 3jbni bezzaf, l'ambiance mzyana w l'prix m3qol 👍"
4,5,web,neg,Commande n° 4587 du 03/10/2026 jamais reçue. Contact : service.client@livraison-rapide.ma ... aucune réponse !!!
5,6,tweet,neg,Franchement la pizza était froide et sans goût 👎 @PizzaRabat vous pouvez mieux faire #mauvaisservice
6,7,web,pos,"M. Alami, le gérant, est très aimable. Il nous a offert le thé à la menthe ! Merci."
7,8,sms,neg,"khayb bzf had lmakla, ma3jbnich, 7ta chi haja ma mzyana 😞"
8,9,web,pos,"Rapport qualité/prix : excellent. 2 couscous + 1 jus = 120,50 DH. On reviendra :)"
9,10,tweet,neg,Attente de 45 min pour une table réservée ?!? Nuuuul. #Casablanca


**Étape 2.** Complétez le dictionnaire `MOTIFS_BRUIT` : un motif par type de bruit (réutilisez le catalogue du cours). La cellule suivante compte, pour chaque type, le **nombre d'avis** qui en contiennent au moins un.
- `html` : une balise `<…>` **ou** une entité comme `&amp;`, `&#39;`, `&nbsp;` ;
- `url` : `http://…`, `https://…` **ou** `www.…` ;
- `email`, `mention` (attention aux adresses e-mail !), `hashtag` ;
- `emoji` : la classe de caractères du cours `[\U0001F300-\U0001FAFF☀-➿]` (fournie) ;
- `allongement` : une **lettre** (pas un chiffre) répétée au moins 3 fois — indice : référence arrière `\1` et la classe `[^\W\d_]` (« un caractère de mot qui n'est ni un chiffre ni `_` »).

In [6]:
MOTIFS_BRUIT = {
    "html": r"<[^>]+>|&#?\w+;",
    "url": r"(?:https?://|www\.)\S+",
    "email": r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+",
    "mention": r"(?<!\w)@\w+",
    "hashtag": r"#\w+",
    "emoji": r"[\U0001F300-\U0001FAFF☀-➿]",
    "allongement": r"([^\W\d_])\1{2,}",
}

In [7]:
import re

def compter_bruit(motifs):
    """Nombre d'avis contenant au moins une occurrence de chaque motif (motif vide → 0)."""
    resultats = {}                          # 1) le dictionnaire à remplir

    for nom, motif in motifs.items():       # 2) on prend les motifs un par un
        if motif == "":                     # 3) motif pas encore écrit : 0
            resultats[nom] = 0
            continue
        compteur = 0                        # 4) on compte les avis concernés
        for avis in CORPUS:
            texte = avis["texte"]
            if re.search(motif, texte):     # au moins une occurrence dans cet avis ?
                compteur = compteur + 1

        resultats[nom] = compteur           # 5) on range le résultat sous le nom du motif

    return resultats
bruit = compter_bruit(MOTIFS_BRUIT); #print(bruit)
pd.Series(bruit, name="nb d'avis").to_frame().T

,html,url,email,mention,hashtag,emoji,allongement
nb d'avis,4,3,1,4,6,9,4


In [8]:
# ✅ Vérification (ne pas modifier)

verifier("html : 4 avis (balises ou entités)", lambda: bruit["html"] == 4)
verifier("url : 3 avis (dont www.)", lambda: bruit["url"] == 3)
verifier("email : 1 avis", lambda: bruit["email"] == 1)
verifier("mention : 4 avis (l'@ d'une adresse e-mail n'est pas une mention)", lambda: bruit["mention"] == 4
         and not re.search(MOTIFS_BRUIT["mention"], "service.client@livraison-rapide.ma"))
verifier("hashtag : 6 avis", lambda: bruit["hashtag"] == 6)
verifier("allongement : 4 avis, et « 1000 » n'est pas un allongement", lambda: bruit["allongement"] == 4
         and not re.search(MOTIFS_BRUIT["allongement"], "1000 DH"))

✅ html : 4 avis (balises ou entités)
✅ url : 3 avis (dont www.)
✅ email : 1 avis
✅ mention : 4 avis (l'@ d'une adresse e-mail n'est pas une mention)
✅ hashtag : 6 avis
✅ allongement : 4 avis, et « 1000 » n'est pas un allongement


**Contrôle.** La bibliothèque `emoji` (`emoji.emoji_count(t)` : nombre d'émojis d'un texte) confirme le décompte de la classe de caractères : exécutez la cellule suivante.

In [9]:
# Contrôle avec la bibliothèque emoji (fourni)
avec_emoji = [d["id"] for d in CORPUS if emoji.emoji_count(d["texte"]) > 0]
print("avis avec émoji (bibliothèque emoji) :", len(avec_emoji), avec_emoji)

avis avec émoji (bibliothèque emoji) : 9 [1, 2, 4, 6, 8, 13, 16, 18, 20]


#### 📝 Interprétation 0.1
Quel type de bruit est le plus fréquent ? Citez deux éléments du corpus qui gêneraient un modèle si on les laissait tels quels, et un élément qu'il serait dommage d'effacer pour une analyse de sentiment.

**Votre réponse :**

**Interprétation 0.1.** Le bruit le plus fréquent est celui des **émojis** : 9 avis sur 20 en contiennent, devant les hashtags (6 avis selon le motif `#\w+`, dont un faux positif : l'entité `&#39;` de l'avis 11, qui ressemble à un hashtag) puis le HTML, les mentions et les allongements (4 avis chacun), les liens (3) et les e-mails (1). Deux éléments gêneraient un modèle laissés tels quels : les balises et entités HTML (`<br>`, `&amp;`, `&#39;`), qui produiraient des tokens sans aucun sens, et les liens ou adresses (`https://avis-resto.ma/r/1542`, `service.client@livraison-rapide.ma`), dont chaque occurrence est unique et ne fait que gonfler le vocabulaire. En revanche, il serait dommage d'effacer les émojis (😍, 😡, 👎) : ce sont des indices de polarité très directs, tout comme les allongements et la ponctuation répétée (« Trooop », « Nuuuul », « !!! »), qui marquent l'intensité du sentiment.

---
## Exercice 1 — Nettoyer avec les regex avancées (40 min)

On construit un nettoyeur **étape par étape** : chaque fonction traite **un** type de bruit, puis `nettoyer()` les enchaîne dans un ordre réfléchi. Rappel du cours : on **remplace par un jeton** (`<URL>`, `<USER>`…) plutôt que d'effacer.

### 1.1 ★ HTML : décoder les entités et retirer les balises
Complétez `retirer_html(t)` :
1. décoder les entités avec `html.unescape` (`&amp;` → `&`, `&#39;` → `'`, `&nbsp;` → espace insécable) ;
2. remplacer chaque balise `<…>` par **un espace** (pas par rien : `plats</p><p>Menu` doit donner deux mots) — motif **non gourmand** ;
3. normaliser les espaces avec la fonction `espaces` fournie.

In [10]:
def espaces(t):
    """Remplace toute suite de blancs (espace, tabulation, \n, espace insécable…) par un espace."""
    return re.sub(r"\s+", " ", t).strip()

def retirer_html(t):
    t = html.unescape(t)
    t = re.sub(r"<[^>]*?>", " ", t)
    return espaces(t)

for i in (0, 2, 10, 18):
    print(retirer_html(CORPUS[i]["texte"]))

Trooop bon le tajine & le service !!! 😍😍 Je recommande https://avis-resto.ma/r/1542
Accueil impeccable , plats copieux. Menu à 85 DH, très correct.
L'équipe est au top, aujourd'hui encore un sans-faute. Les desserts sont faits maison.
Plats trop salés ; le serveur a-t-il seulement goûté ? Bof bof.


In [11]:
# ✅ Vérification (ne pas modifier)

verifier("entités décodées", lambda: retirer_html("L&#39;été &amp; le thé") == "L'été & le thé")
verifier("balises remplacées par un espace", lambda: retirer_html("<p>Menu <b>top</b></p><p>Prix</p>") == "Menu top Prix")
verifier("&nbsp; devient un espace normal", lambda: retirer_html("salés&nbsp;; le") == "salés ; le")

✅ entités décodées
✅ balises remplacées par un espace
✅ &nbsp; devient un espace normal


### 1.2 ★ Remplacer par des jetons
Complétez la liste ordonnée `JETONS` : chaque élément est un couple `(motif, remplacement)` appliqué avec `re.sub`, **dans l'ordre de la liste**.
- liens → `" <URL> "` ; e-mails → `" <EMAIL> "` ; mentions → `" <USER> "` ;
- hashtag → le mot seul, sans `#` (`#déçu` → `déçu`) : utilisez un **groupe** et `\1` dans le remplacement ;
- numéros de mobile marocains → `" <TEL> "` (motif v4 du cours, fourni).

In [12]:
JETONS = [
    (r"(?:https?://|www\.)\S+", " <URL> "),
    (r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+", " <EMAIL> "),
    (r"(?<!\w)@\w+", " <USER> "),
    (r"#(\w+)", r"\1"),
    (r"(?<!\d)(?:0|\+212 ?)[67](?:[ -]?\d{2}){4}(?!\d)", " <TEL> "),
]

def remplacer_jetons(t):
    for motif, remplacement in JETONS:
        if motif:                                   # on ignore les motifs encore vides
            t = re.sub(motif, remplacement, t)
    return espaces(t)

for i in (1, 4, 12, 14):
    print(remplacer_jetons(retirer_html(CORPUS[i]["texte"])))

<USER> livraison en retard de 2h... 😡 déçu livraison
Commande n° 4587 du 03/10/2026 jamais reçue. Contact : <EMAIL> ... aucune réponse !!!
Merci <USER> pour l'anniversaire de ma mère 🎂🎉 Tout était parfait !!! <URL>
Personnel souriant, terrasse agréable. Réservation sur <URL> ou au <TEL> .


In [13]:
# ✅ Vérification (ne pas modifier)

verifier("lien → <URL>", lambda: remplacer_jetons("voir https://t.co/Xy12Ab ou www.resto.ma") == "voir <URL> ou <URL>")
verifier("e-mail → <EMAIL> (et pas <USER>)", lambda: remplacer_jetons("écrire à a.b@emsi.ma vite") == "écrire à <EMAIL> vite")
verifier("mention → <USER>", lambda: remplacer_jetons("Merci @RestoAtlas !") == "Merci <USER> !")
verifier("hashtag → mot", lambda: remplacer_jetons("#déçu et #ftour") == "déçu et ftour")
verifier("téléphone → <TEL>", lambda: remplacer_jetons("au 06 12 34 56 78.") == "au <TEL> .")

✅ lien → <URL>
✅ e-mail → <EMAIL> (et pas <USER>)
✅ mention → <USER>
✅ hashtag → mot
✅ téléphone → <TEL>


**Expérience : l'ordre compte.** La cellule suivante (fournie) applique une version **naïve** du motif des mentions, `@\w+` (sans assertion), **avant** celle des e-mails. Observez.

In [14]:
t = "Contact : service.client@livraison-rapide.ma"
naif = re.sub(r"@\w+", " <USER> ", t)
naif = re.sub(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+", " <EMAIL> ", naif)
print("mention naïve d'abord :", espaces(naif))
print("votre version          :", remplacer_jetons(t))

mention naïve d'abord : Contact : service.client <USER> -rapide.ma
votre version          : Contact : <EMAIL>


#### 📝 Interprétation 1.2
Que devient l'adresse e-mail avec la version naïve ? Donnez **deux** façons de corriger le problème. Pourquoi remplacer un lien par `<URL>` plutôt que de l'effacer ?

**Votre réponse :**

**Interprétation 1.2.** Avec la version naïve, `@\w+` reconnaît « @livraison » à l'intérieur de l'adresse e-mail et la remplace par `<USER>` : on obtient « service.client <USER> -rapide.ma », l'adresse est détruite et le motif des e-mails ne peut plus la reconnaître. Il y a deux corrections possibles : (1) traiter les e-mails **avant** les mentions, comme dans notre liste `JETONS` ; (2) rendre le motif des mentions plus strict avec une assertion de lookbehind négatif, `(?<!\w)@\w+`, pour qu'un `@` collé à un caractère de mot ne soit jamais pris pour une mention. Remplacer un lien par `<URL>` plutôt que de l'effacer permet de conserver l'information « il y avait un lien ici » (utile pour repérer du spam ou de la publicité), de garder la structure de la phrase (les mots voisins ne sont pas collés) et de regrouper tous les liens, tous différents, sous un seul token, ce qui réduit le vocabulaire.

### 1.3 ★★ Allongements et ponctuation répétée
Complétez `reduire_repetitions(t)` avec deux `re.sub` et des **références arrière** :
- une **lettre** répétée 3 fois ou plus est réduite à **2** (`Trooop` → `Troop`, `Nuuuul` → `Nuul`) mais **les chiffres ne bougent pas** (`1000 DH`) ;
- un signe `!`, `?` ou `.` répété est réduit à **1** (`!!!` → `!`, `...` → `.`).

In [15]:
def reduire_repetitions(t):
    t = re.sub(r"([^\W\d_])\1{2,}", r"\1\1", t)
    t = re.sub(r"([!?.])\1+", r"\1", t)
    return t

print(reduire_repetitions("Trooop bon !!! Nuuuul ?? 1000 DH... miniii"))

Troop bon ! Nuul ? 1000 DH. minii


In [16]:
# ✅ Vérification (ne pas modifier)

verifier("lettres : Trooop → Troop, Nuuuul → Nuul", lambda: reduire_repetitions("Trooop Nuuuul") == "Troop Nuul")
verifier("chiffres intacts : 1000 DH", lambda: reduire_repetitions("1000 DH") == "1000 DH")
verifier("ponctuation : !!! → ! ; ... → .", lambda: reduire_repetitions("Top !!! Bof...") == "Top ! Bof.")
verifier("deux lettres identiques restent (sshabi, bonne)", lambda: reduire_repetitions("sshabi bonne") == "sshabi bonne")

✅ lettres : Trooop → Troop, Nuuuul → Nuul
✅ chiffres intacts : 1000 DH
✅ ponctuation : !!! → ! ; ... → .
✅ deux lettres identiques restent (sshabi, bonne)


### 1.4 ★★ Les émojis : garder, retirer ou traduire (deux versions)
On écrit **deux versions** du même traitement, pour comparer :
- **Version A — sans bibliothèque** : la classe de caractères du cours `[\U0001F300-\U0001FAFF☀-➿]` et `re.sub` ;
- **Version B — avec la bibliothèque `emoji`** (`import emoji`), qui connaît la liste officielle Unicode (drapeaux, tons de peau, émojis composés…).

**Version A.** Complétez `traiter_emojis(t, mode)` :
- `"garder"` : le texte inchangé ;
- `"retirer"` : chaque émoji est remplacé par un espace ;
- `"jetons"` : chaque émoji devient `<POS>` s'il est dans `EMOJIS_POS`, `<NEG>` s'il est dans `EMOJIS_NEG`, `<EMOJI>` sinon. Utilisez `re.sub` avec une **fonction de remplacement** (elle reçoit l'objet `Match`).

In [17]:
EMOJI = r"[\U0001F300-\U0001FAFF☀-➿]"
EMOJIS_POS = {"😍", "👍", "😋", "🎉", "🎂"}
EMOJIS_NEG = {"😡", "👎", "😞", "😒", "☹"}

def jetonv1(m):
    e = m.group()
    if e in EMOJIS_POS:
        return " <POS> "
    if e in EMOJIS_NEG:
        return " <NEG> "
    return " <EMOJI> "

def traiter_emojis(t, mode="jetons"):
    if mode == "garder":
        return t
    if mode == "retirer":
        return espaces(re.sub(EMOJI, " ", t))
    return espaces(re.sub(EMOJI, jetonv1, t))

for mode in ("garder", "retirer", "jetons"):
    print(f"{mode:8}", traiter_emojis("Top 😍😍 mais lent 😡 🍕", mode))

garder   Top 😍😍 mais lent 😡 🍕
retirer  Top mais lent
jetons   Top <POS> <POS> mais lent <NEG> <EMOJI>


In [18]:
# ✅ Vérification (ne pas modifier)

verifier("retirer", lambda: traiter_emojis("Top 😍😍 mais lent 😡", "retirer") == "Top mais lent")
verifier("jetons POS / NEG", lambda: traiter_emojis("Top 😍😍 mais lent 😡", "jetons") == "Top <POS> <POS> mais lent <NEG>")
verifier("émoji inconnu → <EMOJI>", lambda: traiter_emojis("pizza 🍕", "jetons") == "pizza <EMOJI>")

✅ retirer
✅ jetons POS / NEG
✅ émoji inconnu → <EMOJI>


**Version B — avec la bibliothèque `emoji`.** Complétez `traiter_emojis_lib(t, mode)` avec les fonctions de la bibliothèque :
- `"retirer"` : `emoji.replace_emoji(t, replace=" ")` remplace chaque émoji (même composé) par un espace ;
- `"jetons"` : `emoji.replace_emoji(t, replace=jeton)` où `jeton(e, data)` reçoit **l'émoji complet** `e` (et un dictionnaire d'informations `data`). Avant de chercher `e` dans `EMOJIS_POS` / `EMOJIS_NEG`, retirez les **tons de peau** et le sélecteur `FE0F` avec le motif `TONS` fourni (`👍🏽` → `👍`, `☹️` → `☹`) ;
- `"texte"` : `emoji.demojize(t, delimiters=(" ", " "))` remplace chaque émoji par son **nom anglais** (`👍` → `thumbs_up`) : le sens est gardé sous forme de mot.

In [19]:
TONS = r"[\U0001F3FB-\U0001F3FF\uFE0F]"   # 5 tons de peau + sélecteur de variante FE0F

def jetonv2(e, data):
    base = re.sub(TONS, "", e)
    if base in EMOJIS_POS:
        return " <POS> "
    if base in EMOJIS_NEG:
        return " <NEG> "
    return " <EMOJI> "

def traiter_emojis_lib(t, mode="jetons"):
    if mode == "garder":
        return t
    if mode == "retirer":
        return espaces(emoji.replace_emoji(t, replace=" "))
    if mode == "texte":
        return espaces(emoji.demojize(t, delimiters=(" ", " ")))
    return espaces(emoji.replace_emoji(t, replace=jetonv2))

for mode in ("garder", "retirer", "jetons", "texte"):
    print(f"{mode:8}", traiter_emojis_lib("Top 😍😍 mais froid 👎 🍕 👍🏽", mode))

garder   Top 😍😍 mais froid 👎 🍕 👍🏽
retirer  Top mais froid
jetons   Top <POS> <POS> mais froid <NEG> <EMOJI> <POS>
texte    Top smiling_face_with_heart-eyes smiling_face_with_heart-eyes mais froid thumbs_down pizza thumbs_up_medium_skin_tone


In [20]:
# ✅ Vérification (ne pas modifier)

verifier("B retirer", lambda: traiter_emojis_lib("Top 😍😍 mais lent 😡", "retirer") == "Top mais lent")
verifier("B jetons : le ton de peau ne change pas le sens (👍🏽 → <POS>)", lambda: traiter_emojis_lib("Top 👍🏽 mais 😡", "jetons") == "Top <POS> mais <NEG>")
verifier("B jetons : un drapeau = un seul émoji", lambda: traiter_emojis_lib("Maroc 🇲🇦", "jetons") == "Maroc <EMOJI>")
verifier("B texte : demojize", lambda: traiter_emojis_lib("Top 👍 bof 👎", "texte") == "Top thumbs_up bof thumbs_down")

✅ B retirer
✅ B jetons : le ton de peau ne change pas le sens (👍🏽 → <POS>)
✅ B jetons : un drapeau = un seul émoji
✅ B texte : demojize


**Expérience : A ou B ?** La cellule suivante (fournie) applique les deux versions à un texte contenant un drapeau, un ton de peau, un émoji avec sélecteur `FE0F`, une touche et une famille (émojis reliés par le caractère invisible ZWJ `U+200D`). `ascii()` rend visibles les caractères invisibles qui restent.

In [21]:
t = "Bravo 🇲🇦 ! 👍🏽 ☹️ 1️⃣ 👨‍👩‍👧"
print("A regex :", len(re.findall(EMOJI, t)), "morceaux →", ascii(traiter_emojis(t, "jetons")))
print("B emoji :", emoji.emoji_count(t), "émojis   →", traiter_emojis_lib(t, "jetons"))
print("émojis trouvés par B :", [e["emoji"] for e in emoji.emoji_list(t)])

A regex : 6 morceaux → 'Bravo \U0001f1f2\U0001f1e6 ! <POS> <EMOJI> <NEG> \ufe0f 1\ufe0f\u20e3 <EMOJI> \u200d <EMOJI> \u200d <EMOJI>'
B emoji : 5 émojis   → Bravo <EMOJI> ! <POS> <NEG> <EMOJI> <EMOJI>
émojis trouvés par B : ['🇲🇦', '👍🏽', '☹️', '1️⃣', '👨\u200d👩\u200d👧']


#### 📝 Interprétation 1.4
(a) Sur le corpus, les versions A et B donnent-elles le même résultat ? Et sur le texte de l'expérience ? Expliquez chaque différence (drapeau, ton de peau, `FE0F`, touche, famille).
(b) Pourquoi faut-il retirer les tons de peau avant de chercher l'émoji dans `EMOJIS_POS` ?
(c) Quel est l'intérêt du mode `"texte"` (`demojize`) par rapport au jeton `<EMOJI>` ? Quand préférer la version A ?

**Votre réponse :**

**Interprétation 1.4.** (a) Sur le corpus, les versions A et B donnent exactement le même résultat (la vérification de `nettoyer` le confirme), car tous les émojis du corpus sont de simples caractères isolés situés dans la plage de la classe. Sur le texte de l'expérience, les résultats diffèrent : A trouve 6 « morceaux » et B 5 émojis. Le **drapeau** 🇲🇦 est formé de deux indicateurs régionaux (U+1F1E6 à U+1F1FF), situés avant la plage `U+1F300` : la regex ne le voit pas et il reste tel quel dans le texte, alors que B le remplace par un seul `<EMOJI>`. Pour 👍🏽, A traite le pouce (`<POS>`) et le **ton de peau** (`<EMOJI>`) séparément, alors que B les traite comme un seul émoji (`<POS>`). Pour ☹️, A reconnaît ☹ (`<NEG>`) mais laisse le sélecteur invisible `FE0F`, alors que B remplace l'ensemble par `<NEG>`. La **touche** 1️⃣ (chiffre + `FE0F` + `U+20E3`) n'est pas du tout détectée par A, alors que B la remplace par `<EMOJI>`. Enfin, la **famille** 👨‍👩‍👧 est coupée par A en trois `<EMOJI>` séparés par des `U+200D` (ZWJ) invisibles, alors que B la remplace par un seul `<EMOJI>`. (b) 👍🏽 est la séquence 👍 + un modificateur de ton de peau : cette chaîne n'est pas dans `EMOJIS_POS`, qui ne contient que le 👍 nu, et elle deviendrait `<EMOJI>` au lieu de `<POS>`, alors que la couleur de peau ne change pas la polarité. (c) `demojize` garde le **sens** de chaque émoji sous forme de mot (`thumbs_down`, `pizza`) alors que `<EMOJI>` fait disparaître toute distinction entre les émojis non répertoriés ; on n'a pas non plus à maintenir de listes à la main. En contrepartie, les noms sont en anglais et alourdissent le texte. On préfère la version A quand on veut éviter une dépendance, quand les émojis sont simples et connus, et quand on veut contrôler précisément ce qui est transformé (listes `EMOJIS_POS` et `EMOJIS_NEG`).

### 1.5 ★★ Assembler le nettoyeur
Complétez `nettoyer(t, emojis)` en enchaînant vos fonctions dans cet ordre : **HTML → jetons → répétitions → émojis → espaces**. Le paramètre `outil` choisit la version du traitement des émojis : `"regex"` (version A) ou `"emoji"` (version B). La cellule affiche ensuite chaque avis avant et après nettoyage.

In [22]:
def nettoyer(t, emojis="jetons", outil="regex"):
    t = retirer_html(t)
    t = remplacer_jetons(t)
    t = reduire_repetitions(t)
    if outil == "regex":
        t = traiter_emojis(t, emojis)
    else:
        t = traiter_emojis_lib(t, emojis)
    return espaces(t)

df["propre"] = [nettoyer(d["texte"]) for d in CORPUS]
df[["texte", "propre"]]

,texte,propre
0,Trooop bon le tajine &amp; le service !!! 😍😍 Je recommande <br>https://avis-resto.ma/r/1542,Troop bon le tajine & le service ! <POS> <POS> Je recommande <URL>
1,@CasaFood livraison en retard de 2h... 😡 #déçu #livraison,<USER> livraison en retard de 2h. <NEG> déçu livraison
2,"<p>Accueil <b>impeccable</b>, plats copieux.</p><p>Menu à 85 DH, très correct.</p>","Accueil impeccable , plats copieux. Menu à 85 DH, très correct."
3,"slt, le resto 3jbni bezzaf, l'ambiance mzyana w l'prix m3qol 👍","slt, le resto 3jbni bezzaf, l'ambiance mzyana w l'prix m3qol <POS>"
4,Commande n° 4587 du 03/10/2026 jamais reçue. Contact : service.client@livraison-rapide.ma ... aucune réponse !!!,Commande n° 4587 du 03/10/2026 jamais reçue. Contact : <EMAIL> . aucune réponse !
5,Franchement la pizza était froide et sans goût 👎 @PizzaRabat vous pouvez mieux faire #mauvaisservice,Franchement la pizza était froide et sans goût <NEG> <USER> vous pouvez mieux faire mauvaisservice
6,"M. Alami, le gérant, est très aimable. Il nous a offert le thé à la menthe ! Merci.","M. Alami, le gérant, est très aimable. Il nous a offert le thé à la menthe ! Merci."
7,"khayb bzf had lmakla, ma3jbnich, 7ta chi haja ma mzyana 😞","khayb bzf had lmakla, ma3jbnich, 7ta chi haja ma mzyana <NEG>"
8,"Rapport qualité/prix : excellent. 2 couscous + 1 jus = 120,50 DH. On reviendra :)","Rapport qualité/prix : excellent. 2 couscous + 1 jus = 120,50 DH. On reviendra :)"
9,Attente de 45 min pour une table réservée ?!? Nuuuul. #Casablanca,Attente de 45 min pour une table réservée ?!? Nuul. Casablanca


In [23]:
# ✅ Vérification (ne pas modifier)

verifier("avis 1 nettoyé", lambda: nettoyer(CORPUS[0]["texte"]) ==
         "Troop bon le tajine & le service ! <POS> <POS> Je recommande <URL>")
verifier("avis 5 nettoyé", lambda: nettoyer(CORPUS[4]["texte"]) ==
         "Commande n° 4587 du 03/10/2026 jamais reçue. Contact : <EMAIL> . aucune réponse !")
verifier("option emojis='retirer'", lambda: nettoyer(CORPUS[19]["texte"], "retirer") ==
         "Meilleur msemen de la ville ! <USER> tu dois tester ftour")
verifier("outil='emoji' : même résultat sur le corpus", lambda: all(nettoyer(d["texte"], outil="emoji") == nettoyer(d["texte"]) for d in CORPUS))
verifier("le corpus entier est nettoyé (20 avis)", lambda: len(df["propre"]) == 20 and df["propre"].str.contains("<").any())

✅ avis 1 nettoyé
✅ avis 5 nettoyé
✅ option emojis='retirer'
✅ outil='emoji' : même résultat sur le corpus
✅ le corpus entier est nettoyé (20 avis)


#### 📝 Interprétation 1.5
(a) Pourquoi faut-il remplacer les liens **avant** de réduire les répétitions (pensez à `www.`) ?
(b) Comparez l'avis 5 avant et après : le nettoyage a-t-il créé un problème pour la **segmentation en phrases** ?
(c) Le smiley `:)` de l'avis 9 et le darija de l'avis 4 sont-ils traités ? Que faudrait-il ajouter ?

**Votre réponse :**

**Interprétation 1.5.** (a) Si l'on réduisait les répétitions avant de traiter les liens, « www » (trois fois la lettre « w ») deviendrait « ww » et le motif des liens (`www\.`) ne le reconnaîtrait plus, le lien resterait donc dans le texte, abîmé. Les liens et les adresses doivent être remplacés par des jetons avant toute transformation qui touche aux caractères. (b) Oui : dans l'avis 5, la suspension « ... » est réduite à un seul point, ce qui donne « Contact : <EMAIL> . aucune réponse ! ». Un segmenteur y voit une fin de phrase et coupe, alors que « aucune réponse ! » continue l'idée précédente ; de plus, le point isolé après `<EMAIL>` est lui aussi un faux séparateur. (c) Non, ni l'un ni l'autre : le smiley `:)` n'est ni un émoji Unicode ni un motif traité, il reste donc dans le texte et sera coupé en `:` et `)` par le tokeniseur ; le darija de l'avis 4 (« 3jbni », « bezzaf », « mzyana », « m3qol ») reste inchangé. Il faudrait ajouter un petit dictionnaire d'émoticônes (`:)` → `<POS>`, `:(` → `<NEG>`) et un lexique ou des règles de normalisation de l'arabizi (par exemple 3 → ع, 7 → ح) pour rapprocher les variantes d'écriture d'un même mot.

---
## Exercice 2 — Tokeniser et segmenter (30 min)

### 2.1 ★ Un tokeniseur regex : les cas particuliers d'abord
Comme dans le cours, le tokeniseur est une **alternative** de motifs essayés **dans l'ordre**. On le construit à partir de morceaux nommés. Complétez les trois morceaux vides :
- `JETON` : nos jetons `<URL>`, `<USER>`, `<POS>`… doivent rester **un seul token** (un `<`, des majuscules, un `>`) ;
- `AUJOURDHUI` : « aujourd'hui » (ou « Aujourd'hui »), qui ne doit **pas** être coupé à l'apostrophe ;
- `ELISION` : des lettres suivies d'une apostrophe (`l'`, `d'`, `qu'`, `L'`) ; acceptez `'` et `’`.

Les morceaux vides sont ignorés : vous pouvez tester au fur et à mesure.

In [24]:
AUJOURDHUI = r"[Aa]ujourd['’]hui"
JETON      = r"<[A-Z]+>"
SIGLE      = r"(?:[A-Z]\.){2,}"     # S.N.C.F.
ABREV      = r"(?:M|Mme|Dr|etc)\."  # abréviations connues
NOMBRE     = r"\d+(?:[.,]\d+)?(?![^\W\d_])"   # 120,50 ; pas le 3 de « 3jbni »
ELISION    = r"[^\W\d_]+['’]"
MOT        = r"\w+(?:-\w+)*"        # mots, avec tirets internes
PONCT      = r"[^\w\s]"             # ponctuation, émojis, symboles

PARTIES = [AUJOURDHUI, JETON, SIGLE, ABREV, NOMBRE, ELISION, MOT, PONCT]   # l'ordre compte !
TOKEN = re.compile("|".join(p for p in PARTIES if p))

def tok_regex(t):
    return TOKEN.findall(t)

print(tok_regex("M. Alami : l'équipe d'aujourd'hui a-t-il payé 120,50 DH ? <URL> <USER> 😍 la S.N.C.F."))

['M.', 'Alami', ':', "l'", 'équipe', "d'", "aujourd'hui", 'a-t-il', 'payé', '120,50', 'DH', '?', '<URL>', '<USER>', '😍', 'la', 'S.N.C.F.']


In [25]:
# ✅ Vérification (ne pas modifier)

verifier("jetons gardés entiers", lambda: tok_regex("via <URL> et <POS>") == ["via", "<URL>", "et", "<POS>"])
verifier("élisions séparées", lambda: tok_regex("L'équipe qu’il") == ["L'", "équipe", "qu’", "il"])
verifier("aujourd'hui en un seul token", lambda: tok_regex("d'aujourd'hui") == ["d'", "aujourd'hui"])
verifier("nombre, sigle, abréviation", lambda: tok_regex("M. X paie 3,5 à la S.N.C.F.") == ["M.", "X", "paie", "3,5", "à", "la", "S.N.C.F."])

✅ jetons gardés entiers
✅ élisions séparées
✅ aujourd'hui en un seul token
✅ nombre, sigle, abréviation


**Expérience.** Que se passe-t-il si l'on place `MOT` **avant** `ELISION` et `AUJOURDHUI` ? Exécutez la cellule (fournie).

In [26]:
mauvais_ordre = re.compile("|".join(p for p in [MOT, AUJOURDHUI, ELISION, JETON, PONCT] if p))
print(mauvais_ordre.findall("l'équipe d'aujourd'hui <URL>"))

['l', "'", 'équipe', 'd', "'", 'aujourd', "'", 'hui', '<URL>']


### 2.2 ★★ Comparer cinq tokeniseurs
Complétez le dictionnaire `TOKENISEURS` (chaque valeur est une fonction `texte → liste de tokens`) :
- `"nltk"` : `word_tokenize(t, language="french")` ;
- `"tweet"` : `TweetTokenizer().tokenize` (tokeniseur de NLTK conçu pour les tweets) ;
- `"spacy"` : les textes des tokens produits par `nlp.tokenizer(t)` (le tokeniseur seul, plus rapide que `nlp(t)`).

La cellule suivante (fournie) compare les tokeniseurs sur deux avis, puis mesure sur tout le corpus nettoyé le **nombre de tokens** et la **taille du vocabulaire** (nombre de tokens différents).

In [27]:
from nltk.tokenize import word_tokenize, sent_tokenize, TweetTokenizer
import spacy
nlp = spacy.load("fr_core_news_sm")

TOKENISEURS = {
    "split": str.split,
    "regex": tok_regex,
    "nltk": lambda t: word_tokenize(t, language="french"),
    "tweet": TweetTokenizer().tokenize,
    "spacy": lambda t: [x.text for x in nlp.tokenizer(t)],
}

In [28]:
for i in (0, 10):
    print("Avis", i + 1, ":", df.loc[i, "propre"])
    for nom, f in TOKENISEURS.items():
        if callable(f):
            print(f"   {nom:6}", f(df.loc[i, "propre"]))

lignes = []
for nom, f in TOKENISEURS.items():
    if callable(f):
        toks = [x for t in df["propre"] for x in f(t)]
        lignes.append({"tokeniseur": nom, "nb tokens": len(toks), "vocabulaire": len(set(toks))})
comparaison = pd.DataFrame(lignes).set_index("tokeniseur")
comparaison

Avis 1 : Troop bon le tajine & le service ! <POS> <POS> Je recommande <URL>
   split  ['Troop', 'bon', 'le', 'tajine', '&', 'le', 'service', '!', '<POS>', '<POS>', 'Je', 'recommande', '<URL>']
   regex  ['Troop', 'bon', 'le', 'tajine', '&', 'le', 'service', '!', '<POS>', '<POS>', 'Je', 'recommande', '<URL>']
   nltk   ['Troop', 'bon', 'le', 'tajine', '&', 'le', 'service', '!', '<', 'POS', '>', '<', 'POS', '>', 'Je', 'recommande', '<', 'URL', '>']
   tweet  ['Troop', 'bon', 'le', 'tajine', '&', 'le', 'service', '!', '<POS>', '<POS>', 'Je', 'recommande', '<URL>']
   spacy  ['Troop', 'bon', 'le', 'tajine', '&', 'le', 'service', '!', '<', 'POS', '>', '<', 'POS', '>', 'Je', 'recommande', '<', 'URL', '>']
Avis 11 : L'équipe est au top, aujourd'hui encore un sans-faute. Les desserts sont faits maison.
   split  ["L'équipe", 'est', 'au', 'top,', "aujourd'hui", 'encore', 'un', 'sans-faute.', 'Les', 'desserts', 'sont', 'faits', 'maison.']
   regex  ["L'", 'équipe', 'est', 'au', 'top', ',', "aujo

,nb tokens,vocabulaire
tokeniseur,,
split,249,197
regex,295,199
nltk,325,195
tweet,289,197
spacy,333,199


In [29]:
# ✅ Vérification (ne pas modifier)

_t = df.loc[0, "propre"]
verifier("5 tokeniseurs utilisables", lambda: all(callable(f) for f in TOKENISEURS.values()))
verifier("nltk = word_tokenize en français", lambda: TOKENISEURS["nltk"](_t) == word_tokenize(_t, language="french"))
verifier("tweet = TweetTokenizer", lambda: TOKENISEURS["tweet"](_t) == TweetTokenizer().tokenize(_t))
verifier("spacy = textes des tokens", lambda: TOKENISEURS["spacy"](_t) == [x.text for x in nlp.tokenizer(_t)])

✅ 5 tokeniseurs utilisables
✅ nltk = word_tokenize en français
✅ tweet = TweetTokenizer
✅ spacy = textes des tokens


#### 📝 Interprétation 2.2
Relevez trois différences entre les tokeniseurs sur les deux avis affichés (jetons `<URL>`/`<POS>`, élision « L'équipe », virgule collée…). Pourquoi `split` donne-t-il le moins de tokens mais pas le plus petit vocabulaire ? Quel tokeniseur choisiriez-vous pour la suite du module, et à quelle condition ?

**Votre réponse :**

**Interprétation 2.2.** Trois différences sont visibles. (1) Les jetons : `split`, notre regex et `TweetTokenizer` gardent `<POS>` et `<URL>` entiers, tandis que NLTK et spaCy les coupent en `<`, `POS`, `>`. (2) L'élision : sur « L'équipe », NLTK et `TweetTokenizer` gardent un seul token « L'équipe », alors que notre regex et spaCy séparent « L' » et « équipe ». (3) La ponctuation collée : `split` laisse « top, », « sans-faute. » et « maison. » avec leur ponctuation, alors que les quatre autres la séparent. `split` donne le moins de tokens (249) mais pas le plus petit vocabulaire (197, contre 195 pour NLTK), parce que la ponctuation collée crée des variantes d'un même mot (« top » et « top, » sont deux entrées) et que ses tokens, moins nombreux, ne compensent pas cette dispersion. Pour la suite du module, je choisirais **notre tokeniseur regex**, car il garde les jetons entiers, sépare les élisions, protège « aujourd'hui », les nombres et les sigles, et son `\w+` fonctionne aussi pour l'arabizi et l'arabe ; à condition que le texte ait été nettoyé avant (jetons en place) et de compléter ses règles si de nouveaux motifs apparaissent. spaCy ne serait préférable que si l'on a besoin de lemmes ou de catégories grammaticales, et seulement après avoir déclaré les jetons comme cas spéciaux du tokeniseur.

### 2.3 ★ Segmenter en phrases
Complétez `segmenter_naif(t)` : couper **après** `.`, `!` ou `?` suivi d'espaces, **sans perdre** la ponctuation (lookbehind, comme dans le cours). La cellule compare ensuite avec NLTK (`sent_tokenize`, modèle Punkt) et spaCy (`doc.sents`).

In [30]:
PARAGRAPHE = ("M. Alami nous a reçus à 20h. Le menu coûte 120,50 DH, etc. et le service est rapide ! "
              "Vous y allez ? Oui.")

def segmenter_naif(t):
    return re.split(r"(?<=[.!?])\s+", t)

segmentations = {
    "naïf (regex)": segmenter_naif(PARAGRAPHE),
    "NLTK (Punkt)": sent_tokenize(PARAGRAPHE, language="french"),
    "spaCy": [s.text for s in nlp(PARAGRAPHE).sents],
}
for nom, phrases in segmentations.items():
    print(f"{nom:13} {len(phrases)} phrases : {phrases}")

naïf (regex)  6 phrases : ['M.', 'Alami nous a reçus à 20h.', 'Le menu coûte 120,50 DH, etc.', 'et le service est rapide !', 'Vous y allez ?', 'Oui.']
NLTK (Punkt)  4 phrases : ['M. Alami nous a reçus à 20h.', 'Le menu coûte 120,50 DH, etc. et le service est rapide !', 'Vous y allez ?', 'Oui.']
spaCy         4 phrases : ['M. Alami nous a reçus à 20h.', 'Le menu coûte 120,50 DH, etc. et le service est rapide !', 'Vous y allez ?', 'Oui.']


In [31]:
# ✅ Vérification (ne pas modifier)

verifier("coupe après . ! ? en gardant la ponctuation", lambda: segmenter_naif("Oui. Non ! Bof ?") == ["Oui.", "Non !", "Bof ?"])
verifier("la règle naïve coupe à tort après « M. » (6 morceaux)", lambda: len(segmenter_naif(PARAGRAPHE)) == 6)

✅ coupe après . ! ? en gardant la ponctuation
✅ la règle naïve coupe à tort après « M. » (6 morceaux)


#### 📝 Interprétation 2.3
Combien de phrases y a-t-il réellement dans `PARAGRAPHE` ? Où la règle naïve se trompe-t-elle, et pourquoi NLTK fait-il mieux ? Comment améliorer la règle naïve sans bibliothèque ?

**Votre réponse :**

**Interprétation 2.3.** Il y a réellement **4 phrases** dans `PARAGRAPHE` : « M. Alami nous a reçus à 20h. », « Le menu coûte 120,50 DH, etc. et le service est rapide ! », « Vous y allez ? » et « Oui. ». La règle naïve en trouve 6 : elle coupe à tort après l'abréviation « M. » et après « etc. », puisqu'elle ne voit qu'un point suivi d'un espace, sans savoir si ce point termine une phrase ou une abréviation. NLTK (Punkt) fait mieux parce que c'est un modèle entraîné qui a appris les abréviations courantes et le contexte d'un point (une majuscule ou une minuscule qui suit, par exemple) ; spaCy donne aussi 4 phrases sur cet exemple. Sans bibliothèque, on peut améliorer la règle de deux façons : protéger d'abord une liste d'abréviations connues (M., Mme, Dr, etc.) en remplaçant temporairement leur point, et exiger une majuscule ou un chiffre après la ponctuation avec une assertion de lookahead (`(?=[A-ZÀ-Ý\d])`), ce qui évite la coupure après « etc. et ».

---
## Exercice 3 ★★ — Livrable : la fonction `preprocess()`

Assemblez votre chaîne dans **une seule fonction** réutilisable dans les TP suivants :
1. `nettoyer(texte, emojis, outil)` ;
2. `tok_regex` ;
3. si `minuscules=True` : passer chaque token en minuscules **sauf les jetons** (`<URL>` doit rester `<URL>`).

In [32]:
def preprocess(texte, emojis="jetons", minuscules=True, outil="regex"):
    """Texte brut → liste de tokens propres."""
    tokens = tok_regex(nettoyer(texte, emojis, outil))
    if minuscules:
        tokens = [x if re.fullmatch(r"<[A-Z]+>", x) else x.lower() for x in tokens]
    return tokens

print(preprocess(CORPUS[1]["texte"]))
print(preprocess(CORPUS[12]["texte"], emojis="retirer"))

['<USER>', 'livraison', 'en', 'retard', 'de', '2h', '.', '<NEG>', 'déçu', 'livraison']
['merci', '<USER>', 'pour', "l'", 'anniversaire', 'de', 'ma', 'mère', 'tout', 'était', 'parfait', '!', '<URL>']


In [33]:
# ✅ Vérification (ne pas modifier)

verifier("avis 2 prétraité", lambda: preprocess(CORPUS[1]["texte"]) ==
         ["<USER>", "livraison", "en", "retard", "de", "2h", ".", "<NEG>", "déçu", "livraison"])
verifier("les jetons restent en majuscules", lambda: preprocess("Voir www.x.ma 😍") == ["voir", "<URL>", "<POS>"])
verifier("minuscules=False garde la casse", lambda: preprocess("Trooop BON", minuscules=False) == ["Troop", "BON"])
verifier("outil='emoji' : 👍🏽 donne un seul jeton", lambda: preprocess("Top 👍🏽 !", outil="emoji") == ["top", "<POS>", "!"])
verifier("tout le corpus passe sans erreur", lambda: all(isinstance(preprocess(d["texte"]), list) for d in CORPUS))

✅ avis 2 prétraité
✅ les jetons restent en majuscules
✅ minuscules=False garde la casse
✅ outil='emoji' : 👍🏽 donne un seul jeton
✅ tout le corpus passe sans erreur


**Bilan chiffré.** La cellule suivante (fournie) mesure l'effet du prétraitement sur la taille du vocabulaire et affiche les tokens les plus fréquents.

In [34]:
brut = [x for d in CORPUS for x in d["texte"].split()]
pret = [x for d in CORPUS for x in preprocess(d["texte"])]
bilan = pd.DataFrame({"nb tokens": [len(brut), len(pret)], "vocabulaire": [len(set(brut)), len(set(pret))]},
                     index=["texte brut + split", "preprocess()"])
print(bilan, "\n")
print(Counter(pret).most_common(15))

                    nb tokens  vocabulaire
texte brut + split        246          213
preprocess()              295          194 

[('.', 18), (',', 15), ('<POS>', 8), ('le', 7), ('!', 7), ("l'", 5), ('<USER>', 4), ('de', 4), ('<NEG>', 4), (':', 4), ('est', 4), ('<URL>', 3), ('à', 3), ('dh', 3), ('prix', 3)]


#### 📝 Interprétation 3
Le vocabulaire a-t-il diminué ? Quels tokens dominent la liste des plus fréquents, et qu'en conclure pour la suite (pensez au TP sur la normalisation : mots vides, lemmes) ?

**Votre réponse :**

**Interprétation 3.** Le vocabulaire a diminué, mais modestement : il passe de 213 à 194 formes différentes (environ 9 % de moins), alors que le nombre de tokens augmente de 246 à 295, puisque la ponctuation collée est maintenant séparée en tokens à part. Les tokens les plus fréquents sont dominés par la ponctuation (« . » 18 fois, « , » 15 fois, « ! » 7 fois) et par des mots grammaticaux (« le », « l' », « de », « est », « à »), qui portent très peu de sens ; viennent ensuite nos jetons `<POS>` (8), `<USER>` et `<NEG>` (4), `<URL>` (3). On en conclut que la suite du module doit filtrer ou pondérer ces tokens : retirer ou pondérer les mots vides et la ponctuation (TF-IDF), et appliquer une lemmatisation pour regrouper les formes d'un même mot (le vocabulaire reste très dispersé, les variantes de darija restant isolées). Les jetons `<POS>` et `<NEG>`, eux, sont des indices utiles pour l'analyse de sentiment et ne doivent pas être éliminés.

---
## ✔️ Avant de rendre
- [ ] Identification complète et nom du fichier conforme
- [ ] `Noyau → Redémarrer et tout exécuter` fonctionne sans erreur
- [ ] Toutes les vérifications ✅ passent (exercices 0 à 3 ; *Pour aller plus loin* facultatif)
- [ ] Toutes les interprétations 📝 sont rédigées (phrases complètes, appuyées sur vos résultats)
- [ ] Les deux versions du traitement des émojis (A : regex, B : bibliothèque `emoji`) passent leurs tests
- [ ] La fonction `preprocess()` est prête : elle servira dans les TP suivants

**Bilan personnel (facultatif) :** ce que j'ai appris / ce qui m'a posé difficulté.

_…_